# Muon Regime Restoration Test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/colab/vit_optimizer_diagnostics/muon_regime_scaling/01_restore_old_muon_regime.ipynb)

현재 실패 run의 **진단량과 Muon 구현은 그대로 유지**하고, 학습 regime만 과거 성공 조건으로 되돌린다.

목표는 accuracy 복구 자체보다 `representation → class organization → function sensitivity → loss geometry`가 함께 old-success 상태로 돌아오는지를 확인하는 것이다.

## 0. 환경 준비

과거 성공 조건: seed 7, train 40k, validation 5k, batch 512, 50 epochs. 현재 메인 폴더의 모델/optimizer/진단 코드를 그대로 호출한다.

In [ ]:
!pip -q install datasets prodigyopt tensorboard scikit-learn

%cd /content
!rm -rf deep-learning-diagnostics-and-improvement
!git clone -q https://github.com/HisameOgasahara/deep-learning-diagnostics-and-improvement.git
%cd /content/deep-learning-diagnostics-and-improvement/colab/vit_optimizer_diagnostics/muon_regime_scaling

import sys
from pathlib import Path

PARENT = Path.cwd().parent
sys.path.insert(0, str(PARENT))
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import torch

from muon_regime_runner import run_muon_regime

## 1. Old-success regime 복원

이 셀에서 바뀌는 것은 학습 regime뿐이다. `run_muon_regime`은 현재 `vit_lab_model_optim.py`, `vit_lab_train.py`, `vit_lab_repr.py`, `vit_lab_landscape.py`, `vit_lab_extended.py`를 호출한다.

In [ ]:
CONFIG = {
    "name": "restore_old_regime",
    "seed": 7,
    "train_samples": 40_000,
    "batch_size": 512,
    "epochs": 50,
    "validation_mode": "old_train_holdout_5k",
}

OUTPUT_ROOT = Path("/content/muon_regime_restore_outputs")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG

In [ ]:
result = run_muon_regime(
    config=CONFIG,
    output_root=OUTPUT_ROOT,
    device=DEVICE,
    dynamics_every=10,
)

summary = pd.DataFrame([result["summary"]])
summary

## 2. 학습동역학

Gradient norm / gradient cosine / update norm / update-to-weight / update cosine / displacement / gradient-noise는 현재 메인 진단과 동일하게 저장된다.

In [ ]:
dynamics = result["dynamics"]

dynamics_summary = dynamics.groupby(
    ["parameter", "epoch"],
    as_index=False,
).agg(
    grad_norm=("grad_norm", "mean"),
    grad_cosine_prev=("grad_cosine_prev", "mean"),
    update_norm=("update_norm", "mean"),
    update_to_weight=("update_to_weight", "mean"),
    update_cosine_prev=("update_cosine_prev", "mean"),
    parameter_displacement=("parameter_displacement", "mean"),
)

dynamics_summary.tail(20)

In [ ]:
result["gradient_noise"][[
    "epoch",
    "parameter",
    "grad_mean_norm",
    "grad_variance_trace",
    "gradient_noise_ratio",
]].tail(20)

## 3. Representation / class organization

복구 여부를 볼 핵심량은 penultimate CKA, effective rank, linear probe, NC1, kNN purity, margin이다.

In [ ]:
result["representation"].query("epoch == 50")[[
    "run",
    "epoch",
    "layer",
    "effective_rank",
    "cka_to_init",
    "linear_probe_accuracy",
]]

In [ ]:
result["class_geometry"].query("epoch == 50")

## 4. Function sensitivity / tangent geometry

In [ ]:
display(result["jacobian_summary"])
display(result["tangent_summary"])
display(result["relative_sharpness"])

## 5. Local loss geometry / manifold / trajectory

Hessian 구간은 runner 내부에서 math SDPA를 강제해 double-backward 오류를 피한다.

In [ ]:
display(result["hessian_summary"])
display(result["manifold"])
display(result["trajectory_summary"])

## 6. 과거 성공 / 현재 실패 / 복원 run 비교

`REFERENCE.md`와 `reference_results.csv`에 과거 커밋, Google Drive reference, 핵심 대조군 값을 별도로 고정했다.

In [ ]:
reference = pd.read_csv("reference_results.csv")

restore = summary.copy()
restore["run_id"] = "restore_old_regime"

comparison_columns = [
    "run_id",
    "seed",
    "train_samples",
    "batch_size",
    "epochs",
    "train_accuracy",
    "val_accuracy",
    "val_loss",
    "penultimate_cka_to_init",
    "penultimate_linear_probe",
    "nc1",
    "knn_purity",
    "margin_mean",
    "hessian_min_ritz",
    "hessian_max_ritz",
    "jacobian_spectral_norm",
    "jacobian_participation_rank",
    "tangent_target_alignment",
    "relative_sharpness",
]

comparison = pd.concat([reference, restore], ignore_index=True, sort=False)
comparison[comparison_columns]

## 7. 판정 기준

복원 run에서 accuracy가 old-success 쪽으로 회복하면서 probe/kNN이 증가하고, NC1이 감소하며, margin이 양수화되고, Jacobian/Hessian의 극단값이 완화되면 **regime restoration 가설**을 강하게 지지한다.

이 단계만으로 N, batch, seed, update budget 중 어느 것이 원인인지는 분리하지 않는다. 그 분리는 다음 scaling notebook에서 한다.